# 02 · Preprocesamiento — LoanSight

Decisiones de imputación, encoding y escalado, materializadas en un `ColumnTransformer` de scikit-learn. **Regla anti-fuga**: el preprocesador se ajusta SOLO con `X_train`.

In [ ]:
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Acceso a los módulos del backend (pipeline de preprocesamiento, etc().)
sys.path.insert(0, str(Path.cwd().parent / "backend"))

DB_PATH = Path.cwd().parent / "data" / "processed" / "loansight.duckdb"
con = duckdb.connect(str(DB_PATH), read_only=True)
print("Warehouse:", DB_PATH.exists())


In [ ]:
# Trae las features de modelado (las 6 del contrato API) + grado + objetivos
df = con.execute('''
    SELECT f.loan_amnt, f.term_months, f.annual_inc, f.dti,
           m.emp_length_years, p.purpose, g.grade,
           f.int_rate, f.loan_status, f.is_default
    FROM FACT_LOANS f
    JOIN DIM_PROPOSITO p ON f.proposito_sk = p.proposito_sk
    JOIN DIM_EMPLEO    m ON f.empleo_sk    = m.empleo_sk
    JOIN DIM_GRADO     g ON f.grado_sk     = g.grado_sk
''').df()
print(df.shape)
df.head()


## 1. Análisis de valores nulos
Define la estrategia de imputación: mediana para numéricas, moda para categóricas.

In [ ]:
feats = ['loan_amnt','term_months','annual_inc','dti','emp_length_years','purpose']
df[feats].isna().sum()

## 2. El pipeline de preprocesamiento
Reutilizamos `build_preprocessor` del backend para garantizar que el notebook y la API apliquen exactamente las mismas transformaciones.

In [ ]:
from app.preprocessing.pipeline import (
    build_preprocessor, NUMERIC_FEATURES, ONEHOT_FEATURES, MODEL_COLUMNS)
print('Numéricas :', NUMERIC_FEATURES)
print('One-hot   :', ONEHOT_FEATURES)
print('Columnas  :', MODEL_COLUMNS)

## 3. Ajuste SOLO sobre train (sin fuga de datos)
Partimos primero y ajustamos el preprocesador con el conjunto de entrenamiento; luego transformamos train y test con el mismo objeto.

In [ ]:
from sklearn.model_selection import train_test_split
reg = df.dropna(subset=['int_rate']).sample(50_000, random_state=42)
X, y = reg[MODEL_COLUMNS], reg['int_rate']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pre = build_preprocessor()
pre.fit(X_train)            # <-- SOLO train
Xtr = pre.transform(X_train)
Xte = pre.transform(X_test)
print('Train transformado:', Xtr.shape)
print('Test  transformado:', Xte.shape)
print('Features de salida:', list(pre.get_feature_names_out()))

Las medias/varianzas del `StandardScaler` provienen solo de train. Verificación: la media de las columnas numéricas en *train* transformado es ~0; en *test* NO es exactamente 0 (correcto, no se ajustó con test).

In [ ]:
n_num = len(NUMERIC_FEATURES)
print('Media train (num):', np.round(Xtr[:, :n_num].mean(axis=0), 3))
print('Media test  (num):', np.round(Xte[:, :n_num].mean(axis=0), 3))

## 4. Demostración de OrdinalEncoder para el grado
Aunque el modelo de producción no usa `grade` (no está en la API), el pipeline soporta encoding ordinal respetando el orden A→G.

In [ ]:
from app.preprocessing.pipeline import GRADE_ORDER
pre_ord = build_preprocessor(
    numeric_features=['loan_amnt','dti'],
    onehot_features=['purpose'],
    ordinal_features=['grade'],
    ordinal_categories=GRADE_ORDER)
sample = df[['loan_amnt','dti','purpose','grade']].dropna().head(5)
out = pre_ord.fit_transform(sample)
print('Orden grados:', GRADE_ORDER[0])
print('Features:', list(pre_ord.get_feature_names_out()))
out

## Conclusiones
- **Numéricas**: imputación por mediana + `StandardScaler`.
- **`purpose`**: imputación por moda + `OneHotEncoder(handle_unknown='ignore')`.
- **`grade`** (demostración): `OrdinalEncoder` con orden A→G.
- Todo encapsulado en un `Pipeline` serializable que se ajusta solo con train, eliminando la fuga de datos.